# 2× 时间叠加的 DRAM 复用分析

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import json

root = Path.home() / "project"
out = root / "aisimulate/experiments/disk/analysis/p_local_8ssd_overlay/assets"
out.mkdir(exist_ok=True)
results = root / "results"
old, = (results / "glm52-1p1d-local-8ssd/20261106T083401.924673Z").glob(
    "*384gb_2x/*/prediction/requests.jsonl")
new, = (results / "glm52-1p1d-local-8ssd-overlay/20261110T012210.568695Z").glob(
    "*384gb_overlay_2x/*/prediction/requests.jsonl")
bundle = root / "data/freeinference_agentic_trace/derived/dynosim-openloop-v1/428f809eef5952012f9c-overlay-2x"
manifest = json.loads((bundle / "manifest.json").read_text())
width = manifest["overlay"]["segment_ms"]
origin = manifest["recipe"]["replay_start_ms"]
day_ms = 86400000
fields = ("input_length", "first_admission_g1_reused_input_tokens",
          "first_admission_g2_direct_reused_input_tokens", "first_admission_g3_reused_input_tokens")

def record(row):
    request = json.loads(row)
    key = int(request["session_id"].removeprefix("request_")) - 1
    return key, tuple(request[k] for k in fields)

with old.open() as stream:
    baseline = dict(map(record, stream))
columns = ("requests", "input_tokens", "old_g2", "new_g2", "net_g2",
           "g2_gain_tokens", "g2_loss_tokens", "g2_gain_requests", "g2_loss_requests",
           "gain_with_g1_loss", "net_g1", "net_g3")
groups = defaultdict(lambda: [0] * len(columns))
with new.open() as stream, (bundle / "source_requests.jsonl").open() as side:
    for line, provenance in zip(stream, side, strict=True):
        key, (n, g1, g2, g3) = record(line)
        meta = json.loads(provenance)
        if key != meta["output_row"]:
            raise ValueError("Overlay report order does not match sidecar")
        length, a, b, c = baseline.pop(meta["parent_output_row"])
        if n != length or n != meta["input_length"]:
            raise ValueError("Paired input lengths differ")
        delta = g2 - b
        values = (1, n, b, g2, delta, max(delta, 0), max(-delta, 0),
                  int(delta > 0), int(delta < 0), int(delta > 0 and g1 < a),
                  g1-a, g3-c)
        t = meta["timestamp"]
        for group in (("all", 0), ("day", t // day_ms + 1),
                      ("hour", t // 3600000 % 24), ("week", t // (7*day_ms) + 1),
                      ("segment", (meta["start_ms"] - origin - t) // width + 1)):
            counts = groups[group]
            for i, value in enumerate(values):
                counts[i] += value
if baseline:
    raise ValueError("Unmatched baseline requests")
rows = [dict(group=g, bucket=k, **dict(zip(columns, counts)),
             net_g2_pp=100*counts[4]/counts[1])
        for (g,k),counts in sorted(groups.items())]
with (out / "metrics.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
summary = {"total": rows[0],
           "top_days": sorted((r for r in rows if r["group"]=="day"), key=lambda r:-r["net_g2"])[:5],
           "weeks": [r for r in rows if r["group"]=="week"],
           "hours": [r for r in rows if r["group"]=="hour"]}
(out / "summary.json").write_text(json.dumps(summary, indent=2))
print(summary["total"])
print([(r["bucket"], r["net_g2"], round(r["net_g2_pp"], 3)) for r in summary["top_days"]])
print([(r["bucket"], round(r["net_g2_pp"], 3)) for r in summary["weeks"]])
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
days = [r for r in rows if r["group"]=="day"]
ax.bar([r["bucket"] for r in days], [r["net_g2"]/1e6 for r in days])
ax.set(xlabel="Overlay day (UTC)", ylabel="Net extra G2 reused tokens (millions)")
fig.tight_layout()
fig.savefig(out / "comparison.png", dpi=160)
